In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFE
from sklearn.feature_selection import mutual_info_classif
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("StealthPhisher2025.csv")
df.iloc[:,-1:] = df.iloc[:,-1:].replace(to_replace ="Phishing", value =0)
df.iloc[:,-1:] = df.iloc[:,-1:].replace(to_replace ="Benign", value =1)
LABEL = df.iloc[:,-1:].columns[0]
df[LABEL] = df[LABEL].apply(lambda x: int(x))
colsAll = df.select_dtypes(include=['float64','int64']).columns
colsAll.append(pd.Index([LABEL]))
df = pd.DataFrame(df[colsAll]).copy()
print('Dataset Shape: ', df.shape)
df.iloc[:,-1:].value_counts()

In [ ]:
df.head(5)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
import pandas as pd
import numpy as np

def calculate_mrmr_ranking(dfMRMR):
    X = dfMRMR.drop(columns=[LABEL])

    # Step 1: Standardize the features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

    # Step 2: Calculate pairwise mutual information
    def mutual_information(x, y):
        return mutual_info_regression(x.values.reshape(-1, 1), y.values.ravel())[0]
    
    n_features = X_scaled.shape[1]
    mutual_info_matrix = np.zeros((n_features, n_features))
    
    for i in range(n_features):
        for j in range(n_features):
            if i == j:
                mutual_info_matrix[i, j] = 0  # Ignore self-relevance
            else:
                mutual_info_matrix[i, j] = mutual_information(X_scaled.iloc[:, i], X_scaled.iloc[:, j])

    # Step 3: Calculate mRMR score for each feature
    scores = {}
    for i, feature in enumerate(X_scaled.columns):
        relevance = np.mean(mutual_info_matrix[i, :])  # Mean relevance to other features
        redundancy = np.mean(mutual_info_matrix[:, i])  # Mean redundancy of this feature
        scores[feature] = relevance - redundancy  # mRMR score

    # Step 4: Rank features based on scores
    ranked_features = pd.Series(scores).sort_values(ascending=False)
    return ranked_features

def calculate_pca_ranking(dfPCA):
    X = dfPCA.drop(columns=[LABEL])  
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)    
    pca = PCA(n_components=X.shape[1])
    pca.fit(X_scaled)
    pca_importance = pd.Series(pca.explained_variance_ratio_, index=X.columns)
    return pca_importance
def plot_feature_ranking(ranking, threshold=0.2, title="Feature Ranking by PCA"):
    print(len(ranking))     
    filtered_ranking = ranking[ranking > threshold]
    print(f"Number of features above threshold: {len(filtered_ranking)}")
    filtered_ranking = filtered_ranking.sort_index()
    plt.figure(figsize=(16, 8))
    plt.plot(filtered_ranking.index, filtered_ranking.values, marker='o', linestyle='-', color='b',linewidth=2.5)
    plt.xlabel("Features",fontsize=15, fontweight='bold')
    plt.ylabel("PCA Explained Variance Ratio",fontsize=15, fontweight='bold')
    plt.xticks(fontsize=14, rotation=75, fontweight='bold')
    plt.yticks(fontsize=14, fontweight='bold')
    plt.grid(False)
    plt.tight_layout()
    name = 'charts\\'+title + '.png'
    plt.savefig(name, format='png', dpi=300, bbox_inches='tight')
    plt.show()
def plot_pie_chart(ranking, threshold=0.2, title="PCA Feature Importance Pie Chart"):
    ranking = ranking.sort_index()
    filtered_ranking = ranking[ranking > threshold]
    print(f"Number of features above threshold: {len(filtered_ranking)}")
    plt.figure(figsize=(10, 10))
    plt.pie(
        filtered_ranking,
        labels=filtered_ranking.index,
        autopct="%1.1f%%",
        startangle=135,
        pctdistance=0.85,
        labeldistance=1.02,
        colors=plt.cm.tab20.colors,
        textprops={'fontweight': 'bold'}
    )
    #plt.title(title, fontsize=16, fontweight='bold')
    plt.grid(False)
    plt.tight_layout()
    name = 'charts\\FS_Final.png'
    plt.savefig(name, format='png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
class_0_df = df[df[LABEL] == 0]
class_1_df = df[df[LABEL] == 1]
class_0_pca = calculate_pca_ranking(class_0_df)
print('Done Benign')
class_1_pca = calculate_pca_ranking(class_1_df)
print('Done Malware')

In [ ]:
plot_feature_ranking(class_0_pca,threshold=0, title="PCA Explained Variance - Legitimate URLs")

In [ ]:
plot_feature_ranking(class_1_pca,threshold=0, title="PCA Explained Variance - Malicious URLs")

In [ ]:
# Combine PCA values for both classes
combined_pca = pd.concat([class_0_pca, class_1_pca], axis=1, keys=["Class 0", "Class 1"])
combined_pca["Average"] = combined_pca.mean(axis=1)
# Plot combined PCA values, sorted by feature names
plot_feature_ranking(combined_pca["Average"], threshold=0,title="Average PCA Explained Variance for Legitimate and Malicious URLS")

In [ ]:
# Plot pie chart for features with average PCA > threshold
plot_pie_chart(combined_pca["Average"], threshold=0.01, title="Combined PCA Importance (Threshold > 0.15)")

In [ ]:
cspca = combined_pca[(combined_pca['Average'] >0)]
print(len(cspca.index))
cspca.index

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score,
    confusion_matrix, roc_curve
)
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import time
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Performance DataFrame
performance = pd.DataFrame(columns=['Name', 'Accuracy', 'Precision', 'Sensitivity', 'F1 Score', 'MCC', 'Markedness', "Youden's J", 'FMI', 'Time','NoOfFeatures'])


X = pd.DataFrame(df[cspca.index]).copy()
y = df[LABEL].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scaling data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def markedness(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return precision + specificity - 1

def youdens_j(y_true, y_pred):
    sensitivity = recall_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return sensitivity + specificity - 1

def fowlkes_mallows_index(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    return np.sqrt(precision * recall)
    
# Build Feed-Forward Neural Network + Wide & Deep Model
def build_wide_deep_model(input_dim):
    wide_input = Input(shape=(input_dim,))
    wide_output = Dense(1, activation='sigmoid')(wide_input)

    deep_input = Input(shape=(input_dim,))
    deep_layer_1 = Dense(64, activation='relu')(deep_input)
    deep_layer_2 = Dense(32, activation='relu')(deep_layer_1)
    deep_output = Dense(1, activation='sigmoid')(deep_layer_2)

    concatenated = Concatenate()([wide_output, deep_output])
    final_output = Dense(1, activation='sigmoid')(concatenated)

    model = Model(inputs=[wide_input, deep_input], outputs=final_output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Initialize Model
model = build_wide_deep_model(X_train_scaled.shape[1])

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Training the model with validation and hyperparameter tuning
start_time = time.time()

history = model.fit(
    [X_train_scaled, X_train_scaled], y_train,
    validation_data=([X_test_scaled, X_test_scaled], y_test),
    epochs=100, batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

end_time = time.time()
elapsed_time = end_time - start_time

# Predicting and calculating performance metrics
y_pred = (model.predict([X_test_scaled, X_test_scaled]) > 0.5).astype("int32")

# Performance Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
markedness =  markedness(y_test, y_pred)
youden_j = youdens_j(y_test, y_pred)
fmi = fowlkes_mallows_index(y_test, y_pred)

# Plotting graphs
# 1. Training vs Validation Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
#plt.title('Training vs Validation Accuracy', fontweight='bold')
plt.xlabel('Epochs', fontsize=15, fontweight='bold')
plt.ylabel('Accuracy', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_epochs.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 2. Training vs Validation Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
#plt.title('Training vs Validation Loss', fontweight='bold')
plt.xlabel('Epochs', fontsize=15, fontweight='bold')
plt.ylabel('Loss', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_loss.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 3. ROC-AUC Curve
fpr, tpr, _ = roc_curve(y_test, model.predict([X_test_scaled, X_test_scaled]))
plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
#plt.title('ROC-AUC Curve', fontweight='bold')
plt.xlabel('False Positive Rate', fontsize=15, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_loss.roc.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 4. Confusion Matrix Heatmap
plt.figure(figsize=(4, 3))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
#plt.title('Confusion Matrix', fontweight='bold')
plt.xlabel('Predicted', fontsize=13, fontweight='bold')
plt.ylabel('Actual', fontsize=13, fontweight='bold')
plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.savefig('charts\\result_cm.roc.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# Store results in the performance DataFrame
new_row = pd.DataFrame([{
    'Name': 'FFNN + Wide & Deep Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youden_j,
    'FMI': fmi,
    'Time': elapsed_time,
    'NoOfFeatures': len(cspca.index)
}])

performance = pd.concat([performance, new_row], ignore_index=True)
performance

In [ ]:
cspca = combined_pca[(combined_pca['Average'] >=0.01)]
print(len(cspca.index))
cspca.index

In [ ]:
X = pd.DataFrame(df[cspca.index]).copy()
y = df[LABEL].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scaling data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def markedness(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return precision + specificity - 1

def youdens_j(y_true, y_pred):
    sensitivity = recall_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return sensitivity + specificity - 1

def fowlkes_mallows_index(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    return np.sqrt(precision * recall)
    
# Build Feed-Forward Neural Network + Wide & Deep Model
def build_wide_deep_model(input_dim):
    wide_input = Input(shape=(input_dim,))
    wide_output = Dense(1, activation='sigmoid')(wide_input)

    deep_input = Input(shape=(input_dim,))
    deep_layer_1 = Dense(64, activation='relu')(deep_input)
    deep_layer_2 = Dense(32, activation='relu')(deep_layer_1)
    deep_output = Dense(1, activation='sigmoid')(deep_layer_2)

    concatenated = Concatenate()([wide_output, deep_output])
    final_output = Dense(1, activation='sigmoid')(concatenated)

    model = Model(inputs=[wide_input, deep_input], outputs=final_output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Initialize Model
model = build_wide_deep_model(X_train_scaled.shape[1])

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Training the model with validation and hyperparameter tuning
start_time = time.time()

history = model.fit(
    [X_train_scaled, X_train_scaled], y_train,
    validation_data=([X_test_scaled, X_test_scaled], y_test),
    epochs=100, batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

end_time = time.time()
elapsed_time = end_time - start_time

# Predicting and calculating performance metrics
y_pred = (model.predict([X_test_scaled, X_test_scaled]) > 0.5).astype("int32")

# Performance Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
markedness =  markedness(y_test, y_pred)
youden_j = youdens_j(y_test, y_pred)
fmi = fowlkes_mallows_index(y_test, y_pred)

# Plotting graphs
# 1. Training vs Validation Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
#plt.title('Training vs Validation Accuracy', fontweight='bold')
plt.xlabel('Epochs', fontsize=15, fontweight='bold')
plt.ylabel('Accuracy', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_epochs.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 2. Training vs Validation Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
#plt.title('Training vs Validation Loss', fontweight='bold')
plt.xlabel('Epochs', fontsize=15, fontweight='bold')
plt.ylabel('Loss', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_loss.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 3. ROC-AUC Curve
fpr, tpr, _ = roc_curve(y_test, model.predict([X_test_scaled, X_test_scaled]))
plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
#plt.title('ROC-AUC Curve', fontweight='bold')
plt.xlabel('False Positive Rate', fontsize=15, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_loss.roc.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 4. Confusion Matrix Heatmap
plt.figure(figsize=(4, 3))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
#plt.title('Confusion Matrix', fontweight='bold')
plt.xlabel('Predicted', fontsize=13, fontweight='bold')
plt.ylabel('Actual', fontsize=13, fontweight='bold')
plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.savefig('charts\\result_cm.roc.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# Store results in the performance DataFrame
new_row = pd.DataFrame([{
    'Name': 'FFNN + Wide & Deep Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youden_j,
    'FMI': fmi,
    'Time': elapsed_time,
    'NoOfFeatures': len(cspca.index)
}])

performance = pd.concat([performance, new_row], ignore_index=True)
performance

In [ ]:
cspca = combined_pca[(combined_pca['Average'] >=0.02)]
print(len(cspca.index))
cspca.index

In [ ]:
X = pd.DataFrame(df[cspca.index]).copy()
y = df[LABEL].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scaling data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def markedness(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return precision + specificity - 1

def youdens_j(y_true, y_pred):
    sensitivity = recall_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return sensitivity + specificity - 1

def fowlkes_mallows_index(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    return np.sqrt(precision * recall)
    
# Build Feed-Forward Neural Network + Wide & Deep Model
def build_wide_deep_model(input_dim):
    wide_input = Input(shape=(input_dim,))
    wide_output = Dense(1, activation='sigmoid')(wide_input)

    deep_input = Input(shape=(input_dim,))
    deep_layer_1 = Dense(64, activation='relu')(deep_input)
    deep_layer_2 = Dense(32, activation='relu')(deep_layer_1)
    deep_output = Dense(1, activation='sigmoid')(deep_layer_2)

    concatenated = Concatenate()([wide_output, deep_output])
    final_output = Dense(1, activation='sigmoid')(concatenated)

    model = Model(inputs=[wide_input, deep_input], outputs=final_output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Initialize Model
model = build_wide_deep_model(X_train_scaled.shape[1])

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Training the model with validation and hyperparameter tuning
start_time = time.time()

history = model.fit(
    [X_train_scaled, X_train_scaled], y_train,
    validation_data=([X_test_scaled, X_test_scaled], y_test),
    epochs=100, batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

end_time = time.time()
elapsed_time = end_time - start_time

# Predicting and calculating performance metrics
y_pred = (model.predict([X_test_scaled, X_test_scaled]) > 0.5).astype("int32")

# Performance Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
markedness =  markedness(y_test, y_pred)
youden_j = youdens_j(y_test, y_pred)
fmi = fowlkes_mallows_index(y_test, y_pred)

# Plotting graphs
# 1. Training vs Validation Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
#plt.title('Training vs Validation Accuracy', fontweight='bold')
plt.xlabel('Epochs', fontsize=15, fontweight='bold')
plt.ylabel('Accuracy', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_epochs.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 2. Training vs Validation Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
#plt.title('Training vs Validation Loss', fontweight='bold')
plt.xlabel('Epochs', fontsize=15, fontweight='bold')
plt.ylabel('Loss', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_loss.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 3. ROC-AUC Curve
fpr, tpr, _ = roc_curve(y_test, model.predict([X_test_scaled, X_test_scaled]))
plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
#plt.title('ROC-AUC Curve', fontweight='bold')
plt.xlabel('False Positive Rate', fontsize=15, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_loss.roc.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 4. Confusion Matrix Heatmap
plt.figure(figsize=(4, 3))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
#plt.title('Confusion Matrix', fontweight='bold')
plt.xlabel('Predicted', fontsize=13, fontweight='bold')
plt.ylabel('Actual', fontsize=13, fontweight='bold')
plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.savefig('charts\\result_cm.roc.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# Store results in the performance DataFrame
new_row = pd.DataFrame([{
    'Name': 'FFNN + Wide & Deep Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youden_j,
    'FMI': fmi,
    'Time': elapsed_time,
    'NoOfFeatures': len(cspca.index)
}])

performance = pd.concat([performance, new_row], ignore_index=True)
performance

In [ ]:
cspca = combined_pca[(combined_pca['Average'] >=0.03)]
print(len(cspca.index))
cspca.index

In [ ]:
X = pd.DataFrame(df[cspca.index]).copy()
y = df[LABEL].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scaling data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def markedness(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return precision + specificity - 1

def youdens_j(y_true, y_pred):
    sensitivity = recall_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return sensitivity + specificity - 1

def fowlkes_mallows_index(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    return np.sqrt(precision * recall)
    
# Build Feed-Forward Neural Network + Wide & Deep Model
def build_wide_deep_model(input_dim):
    wide_input = Input(shape=(input_dim,))
    wide_output = Dense(1, activation='sigmoid')(wide_input)

    deep_input = Input(shape=(input_dim,))
    deep_layer_1 = Dense(64, activation='relu')(deep_input)
    deep_layer_2 = Dense(32, activation='relu')(deep_layer_1)
    deep_output = Dense(1, activation='sigmoid')(deep_layer_2)

    concatenated = Concatenate()([wide_output, deep_output])
    final_output = Dense(1, activation='sigmoid')(concatenated)

    model = Model(inputs=[wide_input, deep_input], outputs=final_output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Initialize Model
model = build_wide_deep_model(X_train_scaled.shape[1])

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Training the model with validation and hyperparameter tuning
start_time = time.time()

history = model.fit(
    [X_train_scaled, X_train_scaled], y_train,
    validation_data=([X_test_scaled, X_test_scaled], y_test),
    epochs=100, batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

end_time = time.time()
elapsed_time = end_time - start_time

# Predicting and calculating performance metrics
y_pred = (model.predict([X_test_scaled, X_test_scaled]) > 0.5).astype("int32")

# Performance Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
markedness =  markedness(y_test, y_pred)
youden_j = youdens_j(y_test, y_pred)
fmi = fowlkes_mallows_index(y_test, y_pred)

# Plotting graphs
# 1. Training vs Validation Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
#plt.title('Training vs Validation Accuracy', fontweight='bold')
plt.xlabel('Epochs', fontsize=15, fontweight='bold')
plt.ylabel('Accuracy', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_epochs.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 2. Training vs Validation Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
#plt.title('Training vs Validation Loss', fontweight='bold')
plt.xlabel('Epochs', fontsize=15, fontweight='bold')
plt.ylabel('Loss', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_loss.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 3. ROC-AUC Curve
fpr, tpr, _ = roc_curve(y_test, model.predict([X_test_scaled, X_test_scaled]))
plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
#plt.title('ROC-AUC Curve', fontweight='bold')
plt.xlabel('False Positive Rate', fontsize=15, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_loss.roc.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 4. Confusion Matrix Heatmap
plt.figure(figsize=(4, 3))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
#plt.title('Confusion Matrix', fontweight='bold')
plt.xlabel('Predicted', fontsize=13, fontweight='bold')
plt.ylabel('Actual', fontsize=13, fontweight='bold')
plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.savefig('charts\\result_cm.roc.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# Store results in the performance DataFrame
new_row = pd.DataFrame([{
    'Name': 'FFNN + Wide & Deep Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youden_j,
    'FMI': fmi,
    'Time': elapsed_time,
    'NoOfFeatures': len(cspca.index)
}])

performance = pd.concat([performance, new_row], ignore_index=True)
performance

In [ ]:
cspca = combined_pca[(combined_pca['Average'] >=0.04)]
print(len(cspca.index))
cspca.index

In [ ]:
X = pd.DataFrame(df[cspca.index]).copy()
y = df[LABEL].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scaling data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def markedness(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return precision + specificity - 1

def youdens_j(y_true, y_pred):
    sensitivity = recall_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return sensitivity + specificity - 1

def fowlkes_mallows_index(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    return np.sqrt(precision * recall)
    
# Build Feed-Forward Neural Network + Wide & Deep Model
def build_wide_deep_model(input_dim):
    wide_input = Input(shape=(input_dim,))
    wide_output = Dense(1, activation='sigmoid')(wide_input)

    deep_input = Input(shape=(input_dim,))
    deep_layer_1 = Dense(64, activation='relu')(deep_input)
    deep_layer_2 = Dense(32, activation='relu')(deep_layer_1)
    deep_output = Dense(1, activation='sigmoid')(deep_layer_2)

    concatenated = Concatenate()([wide_output, deep_output])
    final_output = Dense(1, activation='sigmoid')(concatenated)

    model = Model(inputs=[wide_input, deep_input], outputs=final_output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Initialize Model
model = build_wide_deep_model(X_train_scaled.shape[1])

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Training the model with validation and hyperparameter tuning
start_time = time.time()

history = model.fit(
    [X_train_scaled, X_train_scaled], y_train,
    validation_data=([X_test_scaled, X_test_scaled], y_test),
    epochs=100, batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

end_time = time.time()
elapsed_time = end_time - start_time

# Predicting and calculating performance metrics
y_pred = (model.predict([X_test_scaled, X_test_scaled]) > 0.5).astype("int32")

# Performance Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
markedness =  markedness(y_test, y_pred)
youden_j = youdens_j(y_test, y_pred)
fmi = fowlkes_mallows_index(y_test, y_pred)

# Plotting graphs
# 1. Training vs Validation Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
#plt.title('Training vs Validation Accuracy', fontweight='bold')
plt.xlabel('Epochs', fontsize=15, fontweight='bold')
plt.ylabel('Accuracy', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_epochs.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 2. Training vs Validation Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
#plt.title('Training vs Validation Loss', fontweight='bold')
plt.xlabel('Epochs', fontsize=15, fontweight='bold')
plt.ylabel('Loss', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_loss.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 3. ROC-AUC Curve
fpr, tpr, _ = roc_curve(y_test, model.predict([X_test_scaled, X_test_scaled]))
plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
#plt.title('ROC-AUC Curve', fontweight='bold')
plt.xlabel('False Positive Rate', fontsize=15, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_loss.roc.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 4. Confusion Matrix Heatmap
plt.figure(figsize=(4, 3))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
#plt.title('Confusion Matrix', fontweight='bold')
plt.xlabel('Predicted', fontsize=13, fontweight='bold')
plt.ylabel('Actual', fontsize=13, fontweight='bold')
plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.savefig('charts\\result_cm.roc.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# Store results in the performance DataFrame
new_row = pd.DataFrame([{
    'Name': 'FFNN + Wide & Deep Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youden_j,
    'FMI': fmi,
    'Time': elapsed_time,
    'NoOfFeatures': len(cspca.index)
}])

performance = pd.concat([performance, new_row], ignore_index=True)
performance

In [ ]:
cspca = combined_pca[(combined_pca['Average'] >=0.05)]
print(len(cspca.index))
cspca.index

In [ ]:
X = pd.DataFrame(df[cspca.index]).copy()
y = df[LABEL].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scaling data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def markedness(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return precision + specificity - 1

def youdens_j(y_true, y_pred):
    sensitivity = recall_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return sensitivity + specificity - 1

def fowlkes_mallows_index(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    return np.sqrt(precision * recall)
    
# Build Feed-Forward Neural Network + Wide & Deep Model
def build_wide_deep_model(input_dim):
    wide_input = Input(shape=(input_dim,))
    wide_output = Dense(1, activation='sigmoid')(wide_input)

    deep_input = Input(shape=(input_dim,))
    deep_layer_1 = Dense(64, activation='relu')(deep_input)
    deep_layer_2 = Dense(32, activation='relu')(deep_layer_1)
    deep_output = Dense(1, activation='sigmoid')(deep_layer_2)

    concatenated = Concatenate()([wide_output, deep_output])
    final_output = Dense(1, activation='sigmoid')(concatenated)

    model = Model(inputs=[wide_input, deep_input], outputs=final_output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Initialize Model
model = build_wide_deep_model(X_train_scaled.shape[1])

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Training the model with validation and hyperparameter tuning
start_time = time.time()

history = model.fit(
    [X_train_scaled, X_train_scaled], y_train,
    validation_data=([X_test_scaled, X_test_scaled], y_test),
    epochs=100, batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

end_time = time.time()
elapsed_time = end_time - start_time

# Predicting and calculating performance metrics
y_pred = (model.predict([X_test_scaled, X_test_scaled]) > 0.5).astype("int32")

# Performance Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
markedness =  markedness(y_test, y_pred)
youden_j = youdens_j(y_test, y_pred)
fmi = fowlkes_mallows_index(y_test, y_pred)

# Plotting graphs
# 1. Training vs Validation Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
#plt.title('Training vs Validation Accuracy', fontweight='bold')
plt.xlabel('Epochs', fontsize=15, fontweight='bold')
plt.ylabel('Accuracy', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_epochs.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 2. Training vs Validation Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
#plt.title('Training vs Validation Loss', fontweight='bold')
plt.xlabel('Epochs', fontsize=15, fontweight='bold')
plt.ylabel('Loss', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_loss.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 3. ROC-AUC Curve
fpr, tpr, _ = roc_curve(y_test, model.predict([X_test_scaled, X_test_scaled]))
plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
#plt.title('ROC-AUC Curve', fontweight='bold')
plt.xlabel('False Positive Rate', fontsize=15, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=15, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.legend()
plt.savefig('charts\\result_loss.roc.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# 4. Confusion Matrix Heatmap
plt.figure(figsize=(4, 3))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
#plt.title('Confusion Matrix', fontweight='bold')
plt.xlabel('Predicted', fontsize=13, fontweight='bold')
plt.ylabel('Actual', fontsize=13, fontweight='bold')
plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.savefig('charts\\result_cm.roc.png', format='png', dpi=400, bbox_inches='tight')
plt.show()

# Store results in the performance DataFrame
new_row = pd.DataFrame([{
    'Name': 'FFNN + Wide & Deep Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youden_j,
    'FMI': fmi,
    'Time': elapsed_time,
    'NoOfFeatures': len(cspca.index)
}])

performance = pd.concat([performance, new_row], ignore_index=True)
performance

In [ ]:
performance

In [ ]:
performance.to_csv('combined_pca.csv')